Pol Musté Ferré - 2025/2026

# AutoEncoder for Open Clusters: 4D Latent Space

Convolutional autoencoder that estimates the physical parameters of open clusters from their Gaia colour-magnitude diagrams (CMDs). The model is trained on simulated clusters and then applied to real clusters.

**How it works**

1. Each cluster (a simulated `.parquet` file or a real cluster from the catalogue files) is converted into two 100x100 maps: the CMD star counts and the inverse of the mean G-magnitude error per CMD bin.
2. The convolutional encoder compresses both maps into a 4-dimensional latent vector. The mean parallax and the number of stars of the cluster are added as extra scalar inputs.
3. The decoder reconstructs the CMD star counts from the latent vector (Poisson loss).
4. Four linear heads predict distance, log age, binary fraction and log total mass from the same latent vector.

**Data**

- Simulated clusters: generated with the simulation algorithm provided by Sagar Malhotra.
- Real clusters: 42 open clusters from the Gaia DS3 sample list provided by Sagar Malhotra.

**Notebook sections**

1. Setup: libraries, dataset classes, model, training function and shared helpers.
2. Training and evaluation on simulated clusters.
3. Model testing on individual simulated clusters.
4. Application to real open clusters.

**Repository layout**

| Path | Content |
|---|---|
| `autoencoder_code.ipynb` | This notebook |
| `autoencoder_model.pth` | Trained model (weights and configuration) |
| `data/simulated_data/` | Simulated clusters (`.parquet`) used for training and validation |
| `data/real_data/` | Real cluster catalogues (`masses_q.csv`, `clus_sum.csv`) |
| `plots/` | Figures generated by sections 1 to 3 |
| `plots/cluster_results/` | Figures and results table for the real clusters (section 4) |

Run the notebook cells in order. Section 4 does not need training, only the saved model and the real data. The cluster selection in section 3 can be modified.

## 1. Setup

### 1.1 Libraries and Paths

In [ ]:
import glob
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib.colors import ListedColormap, to_rgba
from scipy.stats import binned_statistic_2d
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

DATA_DIR = 'data/simulated_data/'
REAL_DATA_DIR = 'data/real_data/'
PLOTS_DIR = 'plots/'
RESULTS_DIR = 'plots/cluster_results/'
MODEL_PATH = 'autoencoder_model.pth'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

### 1.2 Dataset Classes

`CMDDataset` holds the processing shared by simulated and real clusters: building the CMD and G-error maps and normalising the model inputs. `ClusterDataset` reads simulated clusters from `.parquet` files. The cluster parameters are read from the file name, which must follow the pattern `<name>_<distance in pc>_<...>_<log age>_..._<binary fraction>.parquet`. The dataset for real clusters (`RealClusterDataset`) is defined in section 4.

In [ ]:
class CMDDataset(Dataset):
    """
    Base class for the cluster datasets. Each cluster is stored as two maps on a
    (BP-RP, G) grid, together with its scalar inputs and its parameter labels:
      - Channel 0: CMD star counts (also the reconstruction target).
      - Channel 1: inverse of the mean G error per CMD bin.
    Subclasses read their own data format and register each cluster with add_cluster().
    """
    def __init__(self, transform=None, grid_size=100, color_range=(-1, 4), g_range=(5, 21)):
        self.transform = transform
        self.grid_size = grid_size
        self.color_range = color_range
        self.g_range = g_range

        self.data_raw = []
        self.err_raw = []

        self.labels_dist = []
        self.labels_age = []
        self.labels_bin = []
        self.labels_mass = []

        self.inputs_plx = []
        self.inputs_nstars = []

    def build_maps(self, color, mag, g_error):
        """Builds the CMD count map and the inverse mean G-error map from per-star values."""
        # CMD star counts. Transposed so that rows are magnitude and columns are colour.
        H_raw, _, _ = np.histogram2d(
            color, mag,
            bins=[self.grid_size, self.grid_size],
            range=[self.color_range, self.g_range]
        )
        H_raw = np.expand_dims(H_raw.T.astype(np.float32), axis=0)

        # Mean G error per CMD bin (NaN where the bin has no stars)
        err_map, _, _, _ = binned_statistic_2d(
            color, mag, values=g_error, statistic='mean',
            bins=[self.grid_size, self.grid_size],
            range=[self.color_range, self.g_range]
        )
        err_map = err_map.T
        empty_mask = np.isnan(err_map)

        # Invert the error so that precise bins get high values. The 0.008 offset caps the maximum at 125.
        err_map = 1.0 / (err_map + 0.008)
        err_map[empty_mask] = 0.0
        err_map = np.expand_dims(err_map.astype(np.float32), axis=0)

        return H_raw, err_map

    def add_cluster(self, H_raw, err_map, dist_kpc, age, bin_frac, mass_log, n_stars, mean_plx):
        self.data_raw.append(H_raw)
        self.err_raw.append(err_map)
        self.labels_dist.append(dist_kpc)
        self.labels_age.append(age)
        self.labels_bin.append(bin_frac)
        self.labels_mass.append(mass_log)
        self.inputs_nstars.append(n_stars)
        self.inputs_plx.append(mean_plx)

    def __len__(self):
        return len(self.data_raw)

    def __getitem__(self, idx):
        target_tensor = torch.tensor(self.data_raw[idx], dtype=torch.float32)
        err_tensor = torch.tensor(self.err_raw[idx], dtype=torch.float32)

        # Shape (2, grid_size, grid_size)
        combined_tensor = torch.cat((target_tensor, err_tensor), dim=0)

        if self.transform:
            combined_tensor = self.transform(combined_tensor)

        target_tensor = combined_tensor[0:1, :, :]
        err_in = combined_tensor[1:2, :, :]

        # Normalise both input channels to [0, 1]. The target keeps the raw counts.
        cmd_in = torch.log1p(target_tensor)
        if cmd_in.max() > 0:
            cmd_in = cmd_in / cmd_in.max()

        if err_in.max() > 0:
            err_in = err_in / err_in.max()

        input_tensor = torch.cat((cmd_in, err_in), dim=0)

        return (
            input_tensor,
            target_tensor,
            torch.tensor(self.labels_dist[idx], dtype=torch.float32),
            torch.tensor(self.labels_age[idx], dtype=torch.float32),
            torch.tensor(self.labels_bin[idx], dtype=torch.float32),
            torch.tensor(self.inputs_plx[idx], dtype=torch.float32),
            torch.tensor(self.inputs_nstars[idx], dtype=torch.float32),
            torch.tensor(self.labels_mass[idx], dtype=torch.float32)
        )


class ClusterDataset(CMDDataset):
    """Simulated clusters, loaded from .parquet files with one row per star."""
    def __init__(self, file_paths, transform=None, grid_size=100, color_range=(-1, 4), g_range=(5, 21)):
        super().__init__(transform, grid_size, color_range, g_range)

        print("Processing files and loading data...")
        for file_path in file_paths:
            parts = os.path.basename(file_path).replace('.parquet', '').split('_')

            # Distance (pc), log age and binary fraction are encoded in the file name
            if len(parts) >= 6:
                try:
                    dist_kpc = float(parts[1]) / 1000.0
                    age = float(parts[3])
                    bin_frac = float(parts[-1])
                except ValueError:
                    continue
            else:
                continue

            result = self.process_cluster(file_path)

            if result is not None:
                H_raw, err_map, n_stars, mean_plx, total_mass_log = result
                self.add_cluster(H_raw, err_map, dist_kpc, age, bin_frac, total_mass_log, n_stars, mean_plx)

    def process_cluster(self, file_path):
        """Converts the stars of one cluster into the model maps and extracts its scalar inputs."""
        try:
            df = pd.read_parquet(file_path)
        except Exception:
            return None

        n_stars = len(df)
        mean_plx = df['parallax'].mean()
        total_mass_log = np.log10(df['total_cluster_mass'].iloc[0])

        req_cols = ["app_BP", "app_RP", "app_G", "app_G_no_error"]
        if not all(col in df.columns for col in req_cols):
            return None

        df = df.dropna(subset=req_cols)
        color = df["app_BP"] - df["app_RP"]
        mag = df["app_G"]
        # G error of each star: difference between the simulated magnitude with and without noise
        g_error = np.abs(df["app_G"] - df["app_G_no_error"])

        H_raw, err_map = self.build_maps(color, mag, g_error)

        return H_raw, err_map, n_stars, mean_plx, total_mass_log

### 1.3 Parameter Distribution Plot

Compares the parameter distributions of the training and validation sets.

In [ ]:
def plot_parameter_distributions(train_dataset, val_dataset):
    """
    Plots the distributions of binary fraction, distance, log age, log mass and
    number of stars for the training and validation sets, and prints summary statistics.
    """
    train_bin  = np.array(train_dataset.labels_bin)
    train_dist = np.array(train_dataset.labels_dist)
    train_age  = np.array(train_dataset.labels_age)
    train_mass = np.array(train_dataset.labels_mass)
    train_n    = np.array(train_dataset.inputs_nstars)

    val_bin    = np.array(val_dataset.labels_bin)
    val_dist   = np.array(val_dataset.labels_dist)
    val_age    = np.array(val_dataset.labels_age)
    val_mass   = np.array(val_dataset.labels_mass)
    val_n      = np.array(val_dataset.inputs_nstars)

    # (name, train values, validation values, colour, x-axis limits)
    params = [
        ('Binary Fraction', train_bin,  val_bin,  '#2ecc71', (0.0,  1.0)),
        ('Distance (kpc)',  train_dist, val_dist, '#3498db', (None, None)),
        ('Log Age',         train_age,  val_age,  '#e67e22', (None, None)),
        ('Log Mass',        train_mass, val_mass, '#9b59b6', (None, None)),
        ('Number of Stars', train_n,    val_n,    '#e74c3c', (None, None)),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for ax, (name, tr, vl, color, xlim) in zip(axes[:5], params):
        all_vals  = np.concatenate([tr, vl])
        bin_edges = np.linspace(all_vals.min(), all_vals.max(), 35)

        ax.hist(tr, bins=bin_edges, alpha=0.65, color=color,
                label=f'Train  (n={len(tr):,})', density=True)
        ax.hist(vl, bins=bin_edges, alpha=0.55, color='#7f8c8d',
                label=f'Val    (n={len(vl):,})', density=True, hatch='//')

        ax.axvline(tr.mean(), color=color,     lw=1.6, ls='--', label=f'Train mean: {tr.mean():.3f}')
        ax.axvline(vl.mean(), color='#555555', lw=1.6, ls=':',  label=f'Val mean:   {vl.mean():.3f}')

        if xlim[0] is not None:
            ax.set_xlim(xlim)

        ax.set_title(name, fontsize=13, fontweight='bold')
        ax.set_xlabel(name, fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    axes[-1].axis('off')

    fig.suptitle('Parameter Distributions - Train vs Validation Set',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR + 'parameter_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n" + "=" * 55)
    print("  Distribution Statistics")
    print("=" * 55)
    for name, tr, vl, *_ in params:
        print(f"\n{name}:")
        print(f"  Train  - mean={tr.mean():.4f}  std={tr.std():.4f}"
              f"  min={tr.min():.4f}  max={tr.max():.4f}")
        print(f"  Val    - mean={vl.mean():.4f}  std={vl.std():.4f}"
              f"  min={vl.min():.4f}  max={vl.max():.4f}")

    print("\n-- Binary Fraction coverage check --")
    for label, arr in [("Train", train_bin), ("Val", val_bin)]:
        low  = (arr < 0.1).sum()
        mid  = ((arr >= 0.1) & (arr < 0.5)).sum()
        high = (arr >= 0.5).sum()
        print(f"  {label}: <0.1 -> {low} ({100*low/len(arr):.1f}%) | "
              f"0.1-0.5 -> {mid} ({100*mid/len(arr):.1f}%) | "
              f">=0.5 -> {high} ({100*high/len(arr):.1f}%)")

### 1.4 Model

Multi-task convolutional autoencoder. The encoder processes the two input channels and its features are concatenated with the mean parallax and the number of stars before being projected to the 4D latent space. The decoder reconstructs the CMD star counts only. Four independent linear heads predict distance, log age, binary fraction and log mass from the latent vector.

In [ ]:
class ClusterAutoencoder4d(nn.Module):
    def __init__(self, grid_size=100, latent_dim=4):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 48, 3, padding=1), nn.ReLU(),
            nn.Conv2d(48, 48, 3, padding=1), nn.ReLU(),
            nn.Conv2d(48, 48, 3, stride=1, padding=1), nn.ReLU(),
            nn.Flatten(),
        )

        dummy = torch.zeros(1, 2, grid_size, grid_size)
        enc_dim = self.encoder(dummy).shape[1]

        # +2 inputs: mean parallax and number of stars
        self.fc_latent = nn.Sequential(
            nn.Linear(enc_dim + 2, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, enc_dim)
        )

        # Parameter heads
        self.distance_head = nn.Linear(latent_dim, 1)
        self.age_head = nn.Linear(latent_dim, 1)
        self.bin_head = nn.Linear(latent_dim, 1)
        self.mass_head = nn.Linear(latent_dim, 1)

        # Softplus keeps the output positive: it is the rate of the Poisson reconstruction
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (48, grid_size // 4, grid_size // 4)),
            nn.ConvTranspose2d(48, 48, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(48, 48, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(48, 32, 3, stride=1, padding=1, output_padding=0), nn.ReLU(),
            nn.ConvTranspose2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Softplus()
        )

    def forward(self, x, parallax, n_stars):
        h = self.encoder(x)

        if parallax.dim() == 1: parallax = parallax.unsqueeze(1)
        if n_stars.dim() == 1: n_stars = n_stars.unsqueeze(1)

        z = self.fc_latent(torch.cat((h, parallax, n_stars), dim=1))
        recon_lambda = self.decoder(self.decoder_fc(z))

        dist_pred = self.distance_head(z).squeeze(1)
        age_pred = self.age_head(z).squeeze(1)
        bin_pred = torch.sigmoid(self.bin_head(z)).squeeze(1)  # binary fraction is bounded in [0, 1]
        mass_pred = self.mass_head(z).squeeze(1)

        return recon_lambda, z, dist_pred, age_pred, bin_pred, mass_pred

### 1.5 Training Function

The total loss is the weighted sum of the Poisson reconstruction loss and the MSE of the four parameter predictions. The learning rate is halved when the validation loss stops improving for 10 epochs.

In [ ]:
def train_autoencoder(model, dataloader, val_loader, weights, epochs=100, lr=1e-3, device='cuda'):
    """Trains the model and returns the per-epoch training and validation loss history."""
    if isinstance(device, str):
        device = torch.device(device)

    model.to(device)

    lat_criterion = nn.MSELoss()
    rec_criterion = nn.PoissonNLLLoss(log_input=False, full=True)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)

    history = {'train_loss': [], 'val_loss': []}

    w_recon, w_dist, w_age, w_bin, w_mass = weights

    for epoch in range(epochs):
        model.train()
        t_loss, t_recon, t_dist, t_age, t_bin, t_mass = 0, 0, 0, 0, 0, 0

        for d_in, d_target, dist_true, age_true, bin_true, plx, n_stars, mass_true in dataloader:
            d_in, d_target = d_in.to(device), d_target.to(device)
            dist_true, age_true, bin_true = dist_true.to(device), age_true.to(device), bin_true.to(device)
            plx, n_stars, mass_true = plx.to(device), n_stars.to(device), mass_true.to(device)

            optimizer.zero_grad()

            recon_lambda, z, dist_pred, age_pred, bin_pred, mass_pred = model(d_in, plx, n_stars)

            l_rec = rec_criterion(recon_lambda, d_target)
            l_dst = lat_criterion(dist_pred, dist_true)
            l_age = lat_criterion(age_pred, age_true)
            l_bin = lat_criterion(bin_pred, bin_true)
            l_mas = lat_criterion(mass_pred, mass_true)

            loss = (l_rec * w_recon) + (l_dst * w_dist) + (l_age * w_age) + (l_bin * w_bin) + (l_mas * w_mass)

            loss.backward()
            optimizer.step()

            t_loss += loss.item()
            t_recon += (l_rec * w_recon).item()
            t_dist += (l_dst * w_dist).item()
            t_age += (l_age * w_age).item()
            t_bin += (l_bin * w_bin).item()
            t_mass += (l_mas * w_mass).item()

        model.eval()
        v_loss, v_recon, v_dist, v_age, v_bin, v_mass = 0, 0, 0, 0, 0, 0

        with torch.no_grad():
            for d_in, d_target, dist_true, age_true, bin_true, plx, n_stars, mass_true in val_loader:
                d_in, d_target = d_in.to(device), d_target.to(device)
                dist_true, age_true, bin_true = dist_true.to(device), age_true.to(device), bin_true.to(device)
                plx, n_stars, mass_true = plx.to(device), n_stars.to(device), mass_true.to(device)

                recon_lambda, z, dist_pred, age_pred, bin_pred, mass_pred = model(d_in, plx, n_stars)

                l_rec = rec_criterion(recon_lambda, d_target)
                l_dst = lat_criterion(dist_pred, dist_true)
                l_age = lat_criterion(age_pred, age_true)
                l_bin = lat_criterion(bin_pred, bin_true)
                l_mas = lat_criterion(mass_pred, mass_true)

                loss = (l_rec * w_recon) + (l_dst * w_dist) + (l_age * w_age) + (l_bin * w_bin) + (l_mas * w_mass)

                v_loss += loss.item()
                v_recon += (l_rec * w_recon).item()
                v_dist += (l_dst * w_dist).item()
                v_age += (l_age * w_age).item()
                v_bin += (l_bin * w_bin).item()
                v_mass += (l_mas * w_mass).item()

        avg_t_loss = t_loss / len(dataloader)
        avg_v_loss = v_loss / len(val_loader)

        history['train_loss'].append(avg_t_loss)
        history['val_loss'].append(avg_v_loss)

        scheduler.step(avg_v_loss)
        current_lr = optimizer.param_groups[0]['lr']

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | LR: {current_lr:.2e}')
            print(f'  Train - Total: {avg_t_loss:.4f} | W_Rec: {t_recon/len(dataloader):.4f} | W_Dist: {t_dist/len(dataloader):.4f} | W_Age: {t_age/len(dataloader):.4f} | W_Bin: {t_bin/len(dataloader):.4f} | W_Mass: {t_mass/len(dataloader):.4f}')
            print(f'  Val   - Total: {avg_v_loss:.4f} | W_Rec: {v_recon/len(val_loader):.4f} | W_Dist: {v_dist/len(val_loader):.4f} | W_Age: {v_age/len(val_loader):.4f} | W_Bin: {v_bin/len(val_loader):.4f} | W_Mass: {v_mass/len(val_loader):.4f}')

    return history

### 1.6 Inference and Plotting Helpers

Functions shared by the model testing (section 3) and the real clusters (section 4).

In [ ]:
def load_model(model_path=MODEL_PATH):
    """Loads the trained model and its configuration from a checkpoint file."""
    checkpoint = torch.load(model_path, map_location=DEVICE)
    config = checkpoint['config']

    model = ClusterAutoencoder4d(
        grid_size=config['grid_size'],
        latent_dim=config['latent_dim']
    ).to(DEVICE)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, config


def predict_cluster(model, dataset, idx=0):
    """
    Runs the model on one cluster of a dataset.
    Returns a dict with the true and predicted parameters, the true CMD and the reconstructed CMD.
    """
    # Add a batch dimension of 1 to every item returned by the dataset
    d_in, d_target, dist_t, age_t, bin_t, plx_t, n_t, mass_t = [t.unsqueeze(0) for t in dataset[idx]]

    with torch.no_grad():
        recon_lambda, _, dist_p, age_p, bin_p, mass_p = model(d_in.to(DEVICE), plx_t.to(DEVICE), n_t.to(DEVICE))

        # The decoder outputs Poisson rates: sample star counts to get a discrete CMD
        recon_img = torch.poisson(recon_lambda).cpu().numpy()[0, 0]

    params = dict(
        dist_true=dist_t.item(), dist_pred=dist_p.item(),
        age_true=age_t.item(),   age_pred=age_p.item(),
        bin_true=bin_t.item(),   bin_pred=bin_p.item(),
        mass_true=mass_t.item(), mass_pred=mass_p.item()
    )
    return params, d_target[0, 0].numpy(), recon_img


# Colormap for the CMD plots: empty bins are shown in dark blue instead of black
_hot_colors = plt.get_cmap('hot', 256)(np.linspace(0, 1, 256))
_hot_colors[0] = to_rgba('darkblue')
CUSTOM_HOT = ListedColormap(_hot_colors)


def plot_cluster_comparison(params, true_img, recon_img, title=None, show_table=False, save_path=None):
    """
    Plots the true CMD, the reconstructed CMD and their difference for one cluster.
    The fourth panel is empty, or a true vs predicted table if show_table is True.
    """
    diff_img = true_img - recon_img
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    if title:
        fig.suptitle(title, fontsize=16, fontweight='bold', y=1.01)

    # Image extent: BP-RP from -1 to 4, G from 21 (bottom) to 5 (top), so bright stars are at the top
    extent = [-1, 4, 21, 5]

    vmax_cmd = max(true_img.max(), recon_img.max())
    if vmax_cmd == 0: vmax_cmd = 1

    def panel_title(name, kind, n_stars):
        return (f"{name}\nDist: {params['dist_' + kind]:.3f} kpc, Age: {params['age_' + kind]:.2f}, "
                f"Bin: {params['bin_' + kind]:.2f}\nLogMass: {params['mass_' + kind]:.3f}, N: {n_stars:.0f}")

    im0 = axes[0, 0].imshow(true_img, cmap=CUSTOM_HOT, aspect='auto', extent=extent, vmin=0, vmax=vmax_cmd)
    axes[0, 0].set_title(panel_title('True (Raw)', 'true', true_img.sum()))
    plt.colorbar(im0, ax=axes[0, 0])

    im1 = axes[0, 1].imshow(recon_img, cmap=CUSTOM_HOT, aspect='auto', extent=extent, vmin=0, vmax=vmax_cmd)
    axes[0, 1].set_title(panel_title('Reconstructed (Discrete)', 'pred', recon_img.sum()))
    plt.colorbar(im1, ax=axes[0, 1])

    vmax_diff = max(abs(diff_img.min()), abs(diff_img.max()))
    if vmax_diff == 0: vmax_diff = 1

    im2 = axes[1, 0].imshow(diff_img, cmap='bwr', aspect='auto', extent=extent, vmin=-vmax_diff, vmax=vmax_diff)
    axes[1, 0].set_title(f'Difference (True - Recon)\nNet Diff: {diff_img.sum():.0f} stars')
    plt.colorbar(im2, ax=axes[1, 0])

    axes[1, 1].axis('off')
    if show_table:
        rows = [(label, f"{params[k + '_true']:.3f}", f"{params[k + '_pred']:.3f}",
                 f"{params[k + '_pred'] - params[k + '_true']:+.3f}")
                for label, k in [('Dist (kpc)', 'dist'), ('LogAge', 'age'),
                                 ('Binary Frac', 'bin'), ('LogMass', 'mass')]]
        table = axes[1, 1].table(
            cellText=rows, colLabels=['Parameter', 'True', 'Predicted', 'Pred - True'],
            cellLoc='center', loc='center', bbox=[0.0, 0.2, 1.0, 0.6]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        for j in range(4):
            table[0, j].set_facecolor('#2c3e50')
            table[0, j].set_text_props(color='white', fontweight='bold')
        axes[1, 1].set_title('True vs Predicted Summary', fontsize=13, pad=12)

    for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
        ax.set_xlabel('Color (BP-RP)')
        ax.set_ylabel('Magnitude (G)')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 2. Training and Evaluation

Runs the full pipeline: data loading, 80/20 train/validation split, training, evaluation plots and model saving. Set the hyperparameters and loss weights at the top of `main()`.

In [ ]:
def main():
    GRID_SIZE = 100
    BATCH_SIZE = 256
    EPOCHS = 250
    LATENT_DIM = 4

    # Loss weights: reconstruction, distance, age, binary fraction, mass
    W_RECON = 1.0
    W_DIST  = 1.0
    W_AGE   = 1.0
    W_BIN   = 1.0
    W_MASS  = 1.0

    LOSS_WEIGHTS = (W_RECON, W_DIST, W_AGE, W_BIN, W_MASS)

    start_time = time.time()
    print(f"Using device: {DEVICE}")

    # ------------------------- Data -------------------------
    file_paths = glob.glob(os.path.join(DATA_DIR, '*.parquet'))
    if not file_paths:
        print(f"No files found in {DATA_DIR}")
        return

    train_files, val_files = train_test_split(file_paths, test_size=0.2, random_state=42)

    print('---Train---')
    train_dataset = ClusterDataset(train_files, grid_size=GRID_SIZE)

    # Input check: CMD and G-error map of the first training cluster
    if len(train_dataset) > 0:
        print("\nPlotting one sample cluster (CMD vs G error map)...")
        sample_cmd_raw = train_dataset.data_raw[0][0]
        sample_err_raw = train_dataset.err_raw[0][0]

        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        im0 = ax[0].imshow(sample_cmd_raw, cmap='hot', aspect='auto')
        ax[0].set_title("CMD (Counts)")
        fig.colorbar(im0, ax=ax[0])

        im1 = ax[1].imshow(sample_err_raw, cmap='viridis', aspect='auto')
        ax[1].set_title("G Error Map (Inverse Mean Error)")
        fig.colorbar(im1, ax=ax[1])
        plt.savefig(PLOTS_DIR + 'input_2d_cluster.png')
        plt.show()

    print('---Validation---')
    val_dataset = ClusterDataset(val_files, grid_size=GRID_SIZE)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    plot_parameter_distributions(train_dataset, val_dataset)

    # ------------------------- Training -------------------------
    model = ClusterAutoencoder4d(grid_size=GRID_SIZE, latent_dim=LATENT_DIM)

    history = train_autoencoder(
        model, train_loader, val_loader, weights=LOSS_WEIGHTS,
        epochs=EPOCHS, lr=1e-3, device=DEVICE
    )

    # ------------------------- Loss history -------------------------
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Val')
    plt.title('Total Loss History (Weighted)')
    plt.xlabel('Epoch')
    plt.yscale('log')
    plt.legend()
    plt.grid(True)
    plt.savefig(PLOTS_DIR + 'loss_function.png')
    plt.show()

    # ------------------------- CMD reconstruction (first validation cluster) -------------------------
    model.eval()
    with torch.no_grad():
        d_in, d_target, dist_t, age_t, bin_t, plx_t, n_t, mass_t = next(iter(val_loader))
        d_in, d_target = d_in.to(DEVICE), d_target.to(DEVICE)
        plx_t, n_t = plx_t.to(DEVICE), n_t.to(DEVICE)

        recon_lambda, _, dist_p, age_p, bin_p, mass_p = model(d_in, plx_t, n_t)

        # The decoder outputs Poisson rates: sample star counts to get a discrete CMD
        discrete_recon = torch.poisson(recon_lambda).cpu().numpy()

        idx = 0
        d_true, d_pred = dist_t[idx].item(), dist_p[idx].item()
        a_true, a_pred = age_t[idx].item(), age_p[idx].item()
        b_true, b_pred = bin_t[idx].item(), bin_p[idx].item()
        m_true, m_pred = mass_t[idx].item(), mass_p[idx].item()

        true_img = d_target[idx, 0].cpu().numpy()
        recon_img = discrete_recon[idx, 0]
        diff_img = true_img - recon_img

        n_true_stars = true_img.sum()
        n_recon_stars = recon_img.sum()

        fig, axes = plt.subplots(2, 2, figsize=(16, 14))

        vmax_cmd = max(true_img.max(), recon_img.max())
        if vmax_cmd == 0: vmax_cmd = 1

        im0 = axes[0, 0].imshow(true_img, cmap='hot', aspect='auto', extent=[-1, 4, 21, 5], vmin=0, vmax=vmax_cmd)
        axes[0, 0].set_title(f'True\nDist: {d_true:.2f}kpc, Age: {a_true:.2f}, Bin: {b_true:.2f}\nLogMass: {m_true:.2f}, N: {n_true_stars:.0f}')
        plt.colorbar(im0, ax=axes[0, 0])

        im1 = axes[0, 1].imshow(recon_img, cmap='hot', aspect='auto', extent=[-1, 4, 21, 5], vmin=0, vmax=vmax_cmd)
        axes[0, 1].set_title(f'Recon\nDist: {d_pred:.2f}kpc, Age: {a_pred:.2f}, Bin: {b_pred:.2f}\nLogMass: {m_pred:.2f}, N: {n_recon_stars:.0f}')
        plt.colorbar(im1, ax=axes[0, 1])

        vmax = max(abs(diff_img.min()), abs(diff_img.max()))
        if vmax == 0: vmax = 1

        im2 = axes[1, 0].imshow(diff_img, cmap='bwr', aspect='auto', extent=[-1, 4, 21, 5], vmin=-vmax, vmax=vmax)
        axes[1, 0].set_title(f'Difference (True - Recon)\nNet Diff: {diff_img.sum():.0f} stars')
        plt.colorbar(im2, ax=axes[1, 0])

        axes[1, 1].axis('off')

        for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
            ax.set_xlabel('Color (BP-RP)')
            ax.set_ylabel('Magnitude (G)')

        plt.tight_layout()
        plt.savefig(PLOTS_DIR + 'cmd_reconstruction.png')
        plt.show()

    # ------------------------- Parameter predictions on the validation set -------------------------
    all_d_true, all_d_pred = [], []
    all_a_true, all_a_pred = [], []
    all_b_true, all_b_pred = [], []
    all_m_true, all_m_pred = [], []

    with torch.no_grad():
        for d_in, d_target, dist_t, age_t, bin_t, plx_t, n_t, mass_t in val_loader:
            d_in, plx_t, n_t = d_in.to(DEVICE), plx_t.to(DEVICE), n_t.to(DEVICE)

            _, _, dist_p, age_p, bin_p, mass_p = model(d_in, plx_t, n_t)

            all_d_true.extend(dist_t.numpy())
            all_d_pred.extend(dist_p.cpu().numpy())
            all_a_true.extend(age_t.numpy())
            all_a_pred.extend(age_p.cpu().numpy())
            all_b_true.extend(bin_t.numpy())
            all_b_pred.extend(bin_p.cpu().numpy())
            all_m_true.extend(mass_t.numpy())
            all_m_pred.extend(mass_p.cpu().numpy())

    d_true_real, d_pred_real = np.array(all_d_true), np.array(all_d_pred)
    a_true_real, a_pred_real = np.array(all_a_true), np.array(all_a_pred)
    b_true_real, b_pred_real = np.array(all_b_true), np.array(all_b_pred)
    m_true_real, m_pred_real = np.array(all_m_true), np.array(all_m_pred)

    mae_d = np.mean(np.abs(d_pred_real - d_true_real))
    mae_a = np.mean(np.abs(a_pred_real - a_true_real))
    mae_b = np.mean(np.abs(b_pred_real - b_true_real))
    mae_m = np.mean(np.abs(m_pred_real - m_true_real))

    print("\n=== Final Results (Mean Absolute Error) ===")
    print(f"Distance MAE: {mae_d:.3f} kpc")
    print(f"Age MAE: {mae_a:.3f}")
    print(f"BinFrac MAE: {mae_b:.3f}")
    print(f"Mass MAE: {mae_m:.3f} (Log10 M_sun)")

    # ------------------------- Predicted vs true -------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))

    LABEL_SIZE = 16
    TITLE_SIZE = 18
    TICK_SIZE = 13

    axes[0, 0].scatter(d_true_real, d_pred_real, alpha=0.5, s=10)
    axes[0, 0].plot([d_true_real.min(), d_true_real.max()], [d_true_real.min(), d_true_real.max()], 'r--')
    axes[0, 0].set_title(f'Distance (kpc)\nMAE: {mae_d:.3f}', fontsize=TITLE_SIZE)
    axes[0, 0].set_xlabel('True Distance', fontsize=LABEL_SIZE)
    axes[0, 0].set_ylabel('Predicted Distance', fontsize=LABEL_SIZE)
    axes[0, 0].tick_params(axis='both', labelsize=TICK_SIZE)
    axes[0, 0].grid(True)

    axes[0, 1].scatter(a_true_real, a_pred_real, alpha=0.5, s=10, c='orange')
    axes[0, 1].plot([a_true_real.min(), a_true_real.max()], [a_true_real.min(), a_true_real.max()], 'r--')
    axes[0, 1].set_title(f'LogAge\nMAE: {mae_a:.3f}', fontsize=TITLE_SIZE)
    axes[0, 1].set_xlabel('True LogAge', fontsize=LABEL_SIZE)
    axes[0, 1].set_ylabel('Predicted LogAge', fontsize=LABEL_SIZE)
    axes[0, 1].tick_params(axis='both', labelsize=TICK_SIZE)
    axes[0, 1].grid(True)

    axes[1, 0].scatter(b_true_real, b_pred_real, alpha=0.5, s=10, c='green')
    axes[1, 0].plot([b_true_real.min(), b_true_real.max()], [b_true_real.min(), b_true_real.max()], 'r--')
    axes[1, 0].set_title(f'BinFrac\nMAE: {mae_b:.3f}', fontsize=TITLE_SIZE)
    axes[1, 0].set_xlabel('True BinFrac', fontsize=LABEL_SIZE)
    axes[1, 0].set_ylabel('Predicted BinFrac', fontsize=LABEL_SIZE)
    axes[1, 0].tick_params(axis='both', labelsize=TICK_SIZE)
    axes[1, 0].grid(True)

    axes[1, 1].scatter(m_true_real, m_pred_real, alpha=0.5, s=10, c='purple')
    axes[1, 1].plot([m_true_real.min(), m_true_real.max()], [m_true_real.min(), m_true_real.max()], 'r--')
    axes[1, 1].set_title(f'Log10(Total Mass)\nMAE: {mae_m:.3f}', fontsize=TITLE_SIZE)
    axes[1, 1].set_xlabel('True Log10(Mass)', fontsize=LABEL_SIZE)
    axes[1, 1].set_ylabel('Predicted Log10(Mass)', fontsize=LABEL_SIZE)
    axes[1, 1].tick_params(axis='both', labelsize=TICK_SIZE)
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR + 'scatter_plots.png')
    plt.show()

    # ------------------------- Relative error: (Pred - True) / True -------------------------
    with np.errstate(divide='ignore', invalid='ignore'):
        d_err = (d_pred_real - d_true_real) / d_true_real
        a_err = (a_pred_real - a_true_real) / a_true_real
        # Binary fraction can be exactly 0: those clusters are set to 0 error to avoid dividing by zero
        b_err = np.divide((b_pred_real - b_true_real), b_true_real, out=np.zeros_like(b_pred_real), where=b_true_real != 0)
        m_err = (m_pred_real - m_true_real) / m_true_real

    fig2, axes2 = plt.subplots(2, 2, figsize=(16, 14))

    axes2[0, 0].scatter(d_true_real, d_err, alpha=0.5, s=10)
    axes2[0, 0].axhline(0, color='r', linestyle='--')
    axes2[0, 0].set_title('Distance: (Predicted - True) / True')
    axes2[0, 0].set_xlabel('True Distance (kpc)')
    axes2[0, 0].set_ylabel('Relative Error')
    axes2[0, 0].grid(True)

    axes2[0, 1].scatter(a_true_real, a_err, alpha=0.5, s=10, c='orange')
    axes2[0, 1].axhline(0, color='r', linestyle='--')
    axes2[0, 1].set_title('LogAge: (Predicted - True) / True')
    axes2[0, 1].set_xlabel('True LogAge')
    axes2[0, 1].set_ylabel('Relative Error')
    axes2[0, 1].grid(True)

    axes2[1, 0].scatter(b_true_real, b_err, alpha=0.5, s=10, c='green')
    axes2[1, 0].axhline(0, color='r', linestyle='--')
    axes2[1, 0].set_title('BinFrac: (Predicted - True) / True')
    axes2[1, 0].set_xlabel('True BinFrac')
    axes2[1, 0].set_ylabel('Relative Error')
    axes2[1, 0].grid(True)

    axes2[1, 1].scatter(m_true_real, m_err, alpha=0.5, s=10, c='purple')
    axes2[1, 1].axhline(0, color='r', linestyle='--')
    axes2[1, 1].set_title('Log10(Total Mass): (Predicted - True) / True')
    axes2[1, 1].set_xlabel('True Log10(Mass)')
    axes2[1, 1].set_ylabel('Relative Error')
    axes2[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR + 'relative_errors.png')
    plt.show()

    # ------------------------- Save model -------------------------
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'config': {'grid_size': GRID_SIZE, 'latent_dim': LATENT_DIM, 'weights': LOSS_WEIGHTS}
    }
    torch.save(checkpoint, MODEL_PATH)
    print(f"\nModel saved to {MODEL_PATH}")

    elapsed_time = time.time() - start_time
    print(f"Execution completed in: {elapsed_time/3600:.2f} hours ({elapsed_time/60:.2f} minutes)")


if __name__ == "__main__":
    main()

## 3. Model Testing on Simulated Clusters

Loads the saved model and compares its predictions with the true parameters of individual simulated clusters. The cluster selection in each cell can be modified, and the plot design in `plot_cluster_comparison` (section 1.6).

In [ ]:
model, config = load_model()

### 3.1 Random Cluster

In [ ]:
all_files = glob.glob(os.path.join(DATA_DIR, '*.parquet'))

if not all_files:
    raise FileNotFoundError(f"No parquet files found in {DATA_DIR}")

selected_file = random.choice(all_files)
print(f"Selected {os.path.basename(selected_file)}")

dataset = ClusterDataset([selected_file], grid_size=config['grid_size'])
params, true_img, recon_img = predict_cluster(model, dataset)
plot_cluster_comparison(params, true_img, recon_img)

### 3.2 All Clusters in an Age and Star-Count Range

Plots every simulated cluster with log age above `TARGET_MIN_AGE` and a number of stars between `MIN_N_STARS` and `MAX_N_STARS`.

In [ ]:
MIN_N_STARS = 120
MAX_N_STARS = 200
TARGET_MIN_AGE = 8.5

all_files = glob.glob(os.path.join(DATA_DIR, '*.parquet'))

if not all_files:
    raise FileNotFoundError(f"No parquet files found in {DATA_DIR}")

print(f"Filtering for log age > {TARGET_MIN_AGE} and stars between {MIN_N_STARS} - {MAX_N_STARS}...")

valid_files = []
for file_path in all_files:
    parts = os.path.basename(file_path).replace('.parquet', '').split('_')

    try:
        current_age = float(parts[3])
    except (IndexError, ValueError):
        continue

    # Filter by age first: it only needs the file name, so it avoids opening most files
    if current_age > TARGET_MIN_AGE:
        current_n = len(pd.read_parquet(file_path, columns=['app_G']))

        if MIN_N_STARS <= current_n <= MAX_N_STARS:
            valid_files.append(file_path)

if not valid_files:
    raise FileNotFoundError("Could not find any files matching the age and star-count range.")

print(f"Found {len(valid_files)} files matching the criteria.\n")

for i, file_path in enumerate(valid_files):
    print(f"--- Processing {i+1}/{len(valid_files)}: {os.path.basename(file_path)} ---")

    dataset = ClusterDataset([file_path], grid_size=config['grid_size'])
    params, true_img, recon_img = predict_cluster(model, dataset)
    plot_cluster_comparison(params, true_img, recon_img)

## 4. Real Open Clusters

Applies the trained model to the 42 real open clusters of the Gaia DS3 sample provided by Sagar Malhotra. No training is done here: the model is only used for inference. Figures and the results table are saved in `plots/cluster_results/`.

**Input files** (in `data/real_data/`)

| File | Content | Columns used |
|---|---|---|
| `masses_q.csv` | One row per star | `Cluster`, `app_BP`, `app_RP`, `app_G`, `parallax`, `mass_A_weighted_mode`, `mass_A_mode`, `q_adopted`, `q_thresh_used` |
| `clus_sum.csv` | One row per cluster | `Cluster`, `mean_dist` (pc), `mean_logage`, `n_stars_binaries`, `n_stars_MS` |

**Differences with the simulated clusters**

- Real stars have no noise-free magnitude (`app_G_no_error`), so the G error of each star is approximated with an analytical function of G.
- The "true" values in the plots are reference values derived from the two catalogues, not ground truth. They are computed in `RealClusterDataset`.

### 4.1 Real Cluster Dataset

In [ ]:
class RealClusterDataset(CMDDataset):
    """
    Real clusters, loaded from the star catalogue (masses_q.csv, one row per star)
    and the cluster summary table (clus_sum.csv, one row per cluster).
    Parameter labels are reference values derived from these two tables.
    """
    def __init__(self, df_masses, df_clus_sum, transform=None, grid_size=100,
                 color_range=(-1, 4), g_range=(5, 21)):
        super().__init__(transform, grid_size, color_range, g_range)
        self.cluster_names = []

        cluster_summary = df_clus_sum.set_index('Cluster').to_dict(orient='index')

        print("Processing clusters from masses_q.csv...")
        for cluster_name, cluster_df in df_masses.groupby('Cluster'):
            if cluster_name not in cluster_summary:
                print(f"  WARNING: '{cluster_name}' not found in clus_sum, skipping.")
                continue
            cs = cluster_summary[cluster_name]

            dist_kpc = cs['mean_dist'] / 1000.0   # pc to kpc
            logage = cs['mean_logage']

            # Binary fraction: stars flagged as binaries over the number of main-sequence stars
            n_ms = cs['n_stars_MS']
            bin_frac = float(cs['n_stars_binaries']) / float(n_ms) if n_ms > 0 else 0.0

            result = self._process_cluster(cluster_df)
            if result is None:
                print(f"  WARNING: could not build the CMD for '{cluster_name}', skipping.")
                continue
            H_raw, err_map, n_stars, mean_plx = result

            self.add_cluster(H_raw, err_map, dist_kpc, logage, bin_frac,
                             self._total_log_mass(cluster_df), n_stars, mean_plx)
            self.cluster_names.append(cluster_name)

        print(f"  {len(self.data_raw)} clusters loaded successfully.\n")

    @staticmethod
    def _total_log_mass(cluster_df):
        """
        log10 of the total cluster mass: sum of the primary masses plus the secondary
        masses of the unresolved binaries.
        """
        # Primary mass: weighted mode, falling back to the plain mode where it is missing
        primary = cluster_df['mass_A_weighted_mode'].fillna(cluster_df['mass_A_mode'])

        # A star is an unresolved binary when its adopted mass ratio q is above the threshold
        # used for that star. Its secondary mass is q times the primary mass.
        valid_q = cluster_df['q_adopted'].notna() & cluster_df['q_thresh_used'].notna()
        is_binary = valid_q & (cluster_df['q_adopted'] > cluster_df['q_thresh_used'])
        secondary_mass = (cluster_df.loc[is_binary, 'q_adopted'] * primary[is_binary]).sum()

        total_mass = primary.dropna().sum() + secondary_mass
        return np.log10(total_mass) if total_mass > 0 else 0.0

    @staticmethod
    def _gaia_g_error(g_mag):
        """
        Approximate G-band photometric error (mag) of Gaia DR3 as a function of G.
        Used in place of the simulated noise, which is not available for real stars:
          sigma_G = 0.0003                     for G < 13
          sigma_G = 0.001 * exp(0.69*(G-13))   for G >= 13
        """
        g = np.asarray(g_mag, dtype=np.float64)
        sigma = np.where(g < 13.0, 3.0e-4, 1.0e-3 * np.exp(0.69 * (g - 13.0)))
        return sigma.astype(np.float32)

    def _process_cluster(self, df):
        """Builds the model maps and scalar inputs from the star rows of one cluster."""
        req_cols = ["app_BP", "app_RP", "app_G", "parallax"]
        if not all(c in df.columns for c in req_cols):
            return None

        df = df.dropna(subset=req_cols)
        if len(df) == 0:
            return None

        color = (df["app_BP"] - df["app_RP"]).values
        mag = df["app_G"].values

        H_raw, err_map = self.build_maps(color, mag, self._gaia_g_error(mag))

        return H_raw, err_map, len(df), float(df['parallax'].mean())

### 4.2 Load Model and Catalogues

In [ ]:
model, config = load_model()
print(f"Model loaded | grid={config['grid_size']}  latent={config['latent_dim']}  device={DEVICE}\n")

df_masses = pd.read_csv(REAL_DATA_DIR + 'masses_q.csv')
df_clus_sum = pd.read_csv(REAL_DATA_DIR + 'clus_sum.csv')
print(f"masses_q : {len(df_masses):,} stars | {df_masses['Cluster'].nunique()} clusters")
print(f"clus_sum : {len(df_clus_sum)} clusters\n")

real_dataset = RealClusterDataset(df_masses, df_clus_sum, grid_size=config['grid_size'])

### 4.3 Inference on All Clusters

Saves one comparison figure per cluster.

In [ ]:
results = []

for idx, cluster_name in enumerate(real_dataset.cluster_names):
    params, true_img, recon_img = predict_cluster(model, real_dataset, idx)
    results.append({'cluster': cluster_name, **params})

    plot_cluster_comparison(
        params, true_img, recon_img,
        title=f'Cluster: {cluster_name}', show_table=True,
        save_path=os.path.join(RESULTS_DIR, f'{cluster_name}_comparison.png')
    )

    print(f"[{idx+1:02d}/{len(real_dataset)}] {cluster_name:<20s} "
          f"dist {params['dist_true']:.2f}->{params['dist_pred']:.2f} kpc | "
          f"age {params['age_true']:.2f}->{params['age_pred']:.2f} | "
          f"bin {params['bin_true']:.3f}->{params['bin_pred']:.3f} | "
          f"logM {params['mass_true']:.2f}->{params['mass_pred']:.2f}")

### 4.4 Summary

Mean absolute error between the predictions and the reference values, saved together with all per-cluster results in `inference_results.csv`.

In [ ]:
df_results = pd.DataFrame(results)
for p in ['dist', 'age', 'bin', 'mass']:
    df_results[f'{p}_err'] = df_results[f'{p}_pred'] - df_results[f'{p}_true']

print(f"=== Mean Absolute Error across {len(df_results)} clusters ===")
for p in ['dist', 'age', 'bin', 'mass']:
    print(f"  {p:<5s} MAE = {df_results[f'{p}_err'].abs().mean():.4f}")

df_results.to_csv(os.path.join(RESULTS_DIR, 'inference_results.csv'), index=False)
print(f"\nResults saved to {RESULTS_DIR}")

### 4.5 Predicted vs Reference Values

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
params_to_plot = [('dist', 'Dist (kpc)'), ('age', 'LogAge'), ('bin', 'BinFrac'), ('mass', 'LogMass')]

for ax, (p, label) in zip(axes, params_to_plot):
    ax.scatter(df_results[f'{p}_true'], df_results[f'{p}_pred'],
               alpha=0.7, edgecolors='k', linewidths=0.5)
    lims = [min(df_results[f'{p}_true'].min(), df_results[f'{p}_pred'].min()),
            max(df_results[f'{p}_true'].max(), df_results[f'{p}_pred'].max())]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel(f'True {label}')
    ax.set_ylabel(f'Predicted {label}')
    ax.set_title(f"{label}  MAE={df_results[f'{p}_err'].abs().mean():.3f}")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'true_vs_pred_scatter.png'), dpi=120)
plt.show()